# DeepShield v2 — MIXED-DATA fine-tune ViT-B/16 (FF++ + real/AI-generated faces)

**Why:** the FF++-only models miss diffusion/chatbot images and new face-swap apps (domain shift). Adding real-vs-AI face datasets
to training widens what the model has seen. Default model here: `vit_b16` (timm vit_base_patch16_224, 224 px).

**Add Input (3 things):**
1. `gradientvoyager/faceforensics-c23-extracted-faces-100k` (FF++, has `dataset_manifest.csv`)
2. One or more *real vs AI-generated face* datasets (folders whose names contain real / fake / ai / generated / diffusion ...).
   Search Kaggle for: "AI generated faces real vs fake", "140k real and fake faces", "diffusion faces", "deepfake faces SDXL".
   Folders it cannot classify are skipped and printed (never guessed).
3. Settings: GPU on, Internet ON. Save Version -> Save & Run All (Commit).

`DS_EXTRA_MAX` (default 15000) caps images per extra dataset/class. Results are reported **per origin** in `eval_report.json`.
After training, re-run the cross-dataset notebook (Celeb-DF) — that, not the test split here, is the real generalisation number.

# DeepShield — fine-tuning (EfficientNet-B4 / ResNet-50 / Xception / ViT-B/16 via ARCH) on FaceForensics++ C23 extracted faces (Kaggle)

Dataset: `gradientvoyager/faceforensics-c23-extracted-faces-100k` (add it via **Add Input**); uses its `dataset_manifest.csv`.
Settings: Accelerator = GPU (T4 x2 or P100), Internet = ON (timm downloads the ImageNet weights).
Run with **Save Version -> Save & Run All (Commit)** so training survives a disconnect.

Outputs (in /kaggle/working): `best.pt`, `efficientnet_b4_state_dict.pth`, `efficientnet_b4.onnx`,
`eval_report.json`, `models_entry_*.json`, `manifest.csv`, `history.json`, `curves.png`.

Rules this notebook enforces: identity-disjoint split, thresholds chosen on validation only, test
set evaluated once, training preprocessing identical to DeepShield inference preprocessing.

In [ ]:
import os, sys, json, math, random, time, hashlib, re, collections, datetime, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, timm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, confusion_matrix,
                             precision_recall_fscore_support, balanced_accuracy_score)
warnings.filterwarnings('ignore', category=UserWarning)

SMOKE = os.environ.get('DS_SMOKE') == '1'          # tiny CPU self-test (never set on Kaggle)
DATA_ROOT = Path(os.environ.get('DS_DATA_ROOT', '/kaggle/input'))
OUT = Path(os.environ.get('DS_OUT', '/kaggle/working')); OUT.mkdir(parents=True, exist_ok=True)

# ---- WHICH MODEL TO TRAIN: change ONLY this line (or set env DS_ARCH) --------------------------------------
ARCH = os.environ.get('DS_ARCH', 'vit_b16')       # efficientnet_b4 | resnet50 | xception | vit_b16 | mesonet4
ARCHS = {   # key = DeepShield model key. img = input size (MUST equal input_size in models.json)
    'efficientnet_b4': dict(timm='efficientnet_b4',        img=320, batch=32, lr=(1e-4, 1e-3), dpr=0.2),
    'resnet50':        dict(timm='resnet50',               img=224, batch=64, lr=(1e-4, 1e-3), dpr=0.05),
    'xception':        dict(timm='legacy_xception',        img=299, batch=32, lr=(1e-4, 1e-3), dpr=None),
    'vit_b16':         dict(timm='vit_base_patch16_224',   img=224, batch=32, lr=(2e-5, 3e-4), dpr=0.1),
    # MesoNet-4 (Afchar et al. 2018): tiny CNN, zero_one input. No ImageNet weights exist for it, so "pretrained" = the
    # AUTHORS' released weights (trained on their own deepfake dataset), fine-tuned here. DS_MESO_INIT=scratch trains from random.
    'mesonet4':        dict(timm=None, img=256, batch=128, lr=(3e-4, 1e-3), dpr=None, epochs=8, norm='zero_one',
                            desc='MesoNet-4 (Afchar 2018)'),
}
assert ARCH in ARCHS, f'ARCH must be one of {list(ARCHS)}'
A = ARCHS[ARCH]; KEY = ARCH
MESO_INIT = os.environ.get('DS_MESO_INIT', 'afchar')      # afchar (fine-tune authors' weights) | scratch
MESO_URL = 'https://raw.githubusercontent.com/DariusAf/MesoNet/master/weights/Meso4_DF.h5'
MESO_H5 = os.environ.get('DS_MESO_H5', '')                # local path override (offline / smoke test)
if ARCH == 'mesonet4' and MESO_INIT == 'scratch':
    A = dict(A, epochs=14, lr=(1e-3, 1e-3)); ARCHS[ARCH] = A

# ---- MIXED-DATA MODE (FF++ + extra real/AI-generated face datasets) -> better generalisation to new generators --------
MIXED = os.environ.get('DS_MIXED', '1') == '1'
EXTRA_MAX = int(os.environ.get('DS_EXTRA_MAX', 15000))   # max images per (extra dataset, class) so one dataset cannot dominate

SEED = 42
IMG = int(os.environ.get('DS_IMG', A['img']))       # input size. MUST equal input_size in models.json
BATCH = int(os.environ.get('DS_BATCH', A['batch'])) # fits T4/P100 16 GB with AMP
EPOCHS = int(os.environ.get('DS_EPOCHS', A.get('epochs', 6)))
WARMUP_EPOCHS = 1
LR_BACKBONE, LR_HEAD = A['lr']; WEIGHT_DECAY = 1e-2
# normalization comes from the pretrained weights' own config (must be one of DeepShield's two modes)
if A.get('norm'):
    NORM = A['norm']
else:
    _mean = tuple(round(float(x), 3) for x in timm.models.get_pretrained_cfg(A['timm']).mean)
    NORM = 'minus_one_one' if _mean == (0.5, 0.5, 0.5) else 'imagenet' if _mean == (0.485, 0.456, 0.406) else None
    assert NORM, f'unsupported pretrained mean {_mean}'
LABEL_SMOOTHING = 0.05
PATIENCE = 3                                         # early stop on validation AUC
SPLIT = (0.70, 0.15, 0.15)                           # train / val / test, by IMAGE count, split by IDENTITY
RESUME = True                                        # continue from last.pt if the session restarts
NUM_WORKERS = min(4, os.cpu_count() or 1)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print(f'torch {torch.__version__} | timm {timm.__version__} | device {DEVICE} | '
      f'{torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU"} | ARCH {ARCH} ({A["timm"] or "custom"}) NORM {NORM} IMG {IMG} BATCH {BATCH} EPOCHS {EPOCHS}')
if DEVICE == 'cpu' and not SMOKE:
    print('WARNING: no GPU. Enable it: Settings -> Accelerator -> GPU. Training on CPU would take days.')

## 1. Locate the dataset + read its manifest (seconds, no folder scan)

The dataset ships `dataset_manifest.csv` (filename, source_id, frame_idx, label, fake_type, split).

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png'}
BASE = DATA_ROOT if DATA_ROOT.exists() else Path('/kaggle/input')
assert BASE.exists(), '/kaggle/input missing: no dataset attached. Right panel -> Add Input -> add the dataset.'
found = []
for depth in range(0, 6):                                   # glob on a literal name: never lists the 100k-image folders
    found = list(BASE.glob('/'.join(['*'] * depth + ['dataset_manifest.csv'])))
    if found: break
if not found:
    print('Contents of', BASE, ':', [c.name for c in sorted(BASE.glob('*'))[:20]])
    raise SystemExit('dataset_manifest.csv not found under /kaggle/input. Is the right dataset attached?')
MANIFEST = found[0]; DS_DIR = MANIFEST.parent
print('manifest:', MANIFEST)
print('dataset dir contains:', sorted(c.name for c in DS_DIR.iterdir()))
man = pd.read_csv(MANIFEST)
print(man.shape); print(man.head(5).to_string())
print(man.label.value_counts().to_dict(), man.fake_type.value_counts().to_dict(), man.split.value_counts().to_dict())

## 2. Resolve file paths, check the split for identity leakage, build the training manifest

In [ ]:
t_ = time.time()
tok = lambda x: str(x).lower()
def patterns(r):
    s_, l_, f_, n_ = r.split, r.label, r.fake_type, r.filename
    for a_ in (s_, tok(s_)):
        for b_ in (l_, tok(l_), f_, tok(f_)):
            yield DS_DIR / a_ / b_ / n_
        yield DS_DIR / a_ / n_
    for b_ in (l_, tok(l_), f_, tok(f_)):
        yield DS_DIR / b_ / r.split / n_
        yield DS_DIR / b_ / n_
probe = man.groupby('fake_type').head(3)
chosen = None
for i_, cand in enumerate(patterns(probe.iloc[0])):         # find ONE pattern that fits all probe rows
    idx = i_
    ok = True
    for r in probe.itertuples():
        pp = list(patterns(r))
        if idx >= len(pp) or not pp[idx].exists(): ok = False; break
    if ok: chosen = idx; break
if chosen is not None:
    man['path'] = [str(list(patterns(r))[chosen]) for r in man.itertuples()]
    print('path pattern #', chosen, '->', man.path.iloc[0])
else:                                                        # fallback: list directories once (names only)
    print('no simple pattern fits; indexing the folders once ...')
    index = {}
    for d, _, files in os.walk(DS_DIR):
        if files:
            index[tuple(x.lower() for x in Path(d).relative_to(DS_DIR).parts)] = (d, set(files))
    paths = []
    for r in man.itertuples():
        hit = None
        for parts, (d, fs) in index.items():
            if tok(r.split) in parts and (tok(r.fake_type) in parts or tok(r.label) in parts) and r.filename in fs:
                hit = str(Path(d) / r.filename); break
        paths.append(hit)
    man['path'] = paths
chk = man[man.path.isna()]
samp = man.dropna(subset=['path']).sample(min(2000, int(man.path.notna().sum())), random_state=0)
missing = pd.concat([chk, samp[~samp.path.map(os.path.exists)]])
if len(missing):
    print(missing.head(5).to_string())
    raise SystemExit(f'{len(missing)} manifest rows have no file on disk (checked a 2000-row sample). Send me this output and the folder listing above.')
print(f'paths resolved in {time.time() - t_:.0f}s')

df = man.copy()
df['label'] = (df.label.str.upper() == 'FAKE').astype(int)             # 0 = real, 1 = fake  (DeepShield fake_index = 1)
df['method'] = np.where(df.label == 0, 'real', df.fake_type)
df['video'] = df.source_id.astype(str) + '/' + df.method
assert set(df.label) == {0, 1}, 'manifest lacks one class'

# identity check on the dataset's own split (source_id = person/video id)
n_sp = df.groupby('source_id').split.nunique()
leaky = int((n_sp > 1).sum())
print(f'source_ids: {len(n_sp)} | ids present in more than one split: {leaky}')
if leaky:
    print('The dataset split is NOT identity-disjoint -> re-splitting by source_id (70/15/15).')
    ids_ = sorted(df.source_id.unique()); random.Random(SEED).shuffle(ids_)
    c1, c2 = int(len(ids_) * SPLIT[0]), int(len(ids_) * (SPLIT[0] + SPLIT[1]))
    amap = {i: ('train' if k < c1 else 'val' if k < c2 else 'test') for k, i in enumerate(ids_)}
    df['split'] = df.source_id.map(amap)
else:
    print('Using the dataset split as shipped (identity-disjoint by source_id).')

df['origin'] = 'ffpp'
if MIXED:
    # Every OTHER dataset attached under /kaggle/input is scanned. A folder is REAL if its path has a real-token and FAKE if it has a
    # fake-token (fake / ai / generated / synthetic / diffusion / gan / deepfake / sd / midjourney ...). Unclassifiable folders are skipped
    # and listed, never guessed.
    FAKE_TOK = re.compile(r'(^|[^a-z])(fake|ai|aigen|generated|synthetic|diffusion|gan|stylegan|deepfake|sdxl|sd|midjourney|dalle|flux|faceswap|swap)([^a-z]|$)', re.I)
    REAL_TOK = re.compile(r'(^|[^a-z])(real|human|authentic|genuine|original|natural|ffhq|celeba)([^a-z]|$)', re.I)
    # manual override for a dataset whose folders carry no class words, e.g. DS_FORCE_LABEL="stable-diffusion=fake,somename=real"
    FORCE_LABEL = {k.strip().lower(): v.strip().lower() for k, v in (x.split('=') for x in os.environ.get('DS_FORCE_LABEL', 'stable-diffusion=fake').split(',') if '=' in x)}
    extra_rows, skipped = [], []
    ds_roots = sorted(p for p in BASE.glob('datasets/*/*') if p.is_dir()) or sorted(p for p in BASE.glob('*') if p.is_dir())   # new / old Kaggle layout
    for root_ in ds_roots:
        if root_ == DS_DIR or DS_DIR.is_relative_to(root_) or root_.is_relative_to(DS_DIR): continue    # skip FF++ itself
        groups = collections.defaultdict(list); t0_ = time.time(); n_seen = 0
        print(f'scanning {root_.relative_to(BASE)} ...', flush=True)
        for d, dn, fn in os.walk(root_):
            imgs_ = [f for f in fn if Path(f).suffix.lower() in IMG_EXT]
            if not imgs_: continue
            n_seen += len(imgs_)
            parts_ = Path(d).relative_to(root_).parts
            lab_f = [x for k_, x in FORCE_LABEL.items() if k_ in root_.name.lower()]      # manual override (DS_FORCE_LABEL)
            if lab_f: f_, r_ = lab_f[0] == 'fake', lab_f[0] == 'real'
            else:                                                  # deepest folder name that names exactly ONE class decides
                f_ = r_ = False
                for part_ in reversed(parts_):
                    a_, b_ = bool(FAKE_TOK.search(part_)), bool(REAL_TOK.search(part_))
                    if a_ != b_: f_, r_ = a_, b_; break
            rel = '/'.join(parts_)
            if f_ == r_: skipped.append((str(Path(d).relative_to(BASE)), len(imgs_), 'unlabeled/ambiguous')); continue
            groups[1 if f_ else 0] += [str(Path(d) / f) for f in imgs_]
            if min(len(groups[0]), len(groups[1])) >= EXTRA_MAX * 4 or n_seen >= EXTRA_MAX * 12 or time.time() - t0_ > 240:
                print(f'  (scan stopped early at {n_seen} images: enough to sample {EXTRA_MAX} per class)'); break
        print(f'  done in {time.time() - t0_:.0f}s: ' + str({k_: len(v_) for k_, v_ in groups.items()}), flush=True)
        for lab_, files_ in groups.items():
            if len(files_) < 200: skipped.append((str(root_.relative_to(BASE)), len(files_), f'too few (label {lab_})')); continue
            files_ = sorted(files_); random.Random(SEED).shuffle(files_); files_ = files_[:EXTRA_MAX]
            name_ = re.sub(r'[^a-z0-9]+', '_', root_.name.lower()).strip('_')
            for k_, fp in enumerate(files_):
                extra_rows.append({'path': fp, 'label': lab_, 'method': (f'extra_{name_}_fake' if lab_ else f'extra_{name_}_real'),
                                   'source_id': f'x_{name_}_{lab_}_{k_}', 'origin': f'extra:{name_}'})
    print('SKIPPED extra folders (check names if one of these should be used):', skipped[:20])
    assert extra_rows, 'DS_MIXED=1 but no extra real/fake datasets found. Add Input -> attach a real-vs-AI face dataset.'
    ex = pd.DataFrame(extra_rows)
    # extras have no identity ids -> split by image, stratified per (method); each file is its own "identity"
    parts = []
    for m_, g_ in ex.groupby('method'):
        g_ = g_.sample(frac=1, random_state=SEED).reset_index(drop=True); n_ = len(g_)
        g_['split'] = np.where(np.arange(n_) < int(n_ * SPLIT[0]), 'train', np.where(np.arange(n_) < int(n_ * (SPLIT[0] + SPLIT[1])), 'val', 'test'))
        parts.append(g_)
    ex = pd.concat(parts, ignore_index=True); ex['video'] = ex.source_id
    print('EXTRA data:'); print(pd.crosstab([ex.origin, ex.label], ex.split))
    df = pd.concat([df, ex], ignore_index=True)
    ytr_ = df[(df.split == 'train')].groupby('origin').label.mean().round(2).to_dict()
    print('fake share in TRAIN per origin:', ytr_)

# optional: fewer fakes in TRAIN only (real:fake is ~1:6) -> faster epochs, better balance. val/test stay untouched.
FAKE_PER_REAL = 2          # None = use every training frame
if FAKE_PER_REAL:
    tr = df[(df.split == 'train') & (df.origin == 'ffpp')]; keep_real = tr[tr.label == 0]
    fk = tr[tr.label == 1]; per_type = int(len(keep_real) * FAKE_PER_REAL / fk.method.nunique())
    fk = pd.concat([g.sample(min(len(g), per_type), random_state=SEED) for _, g in fk.groupby('method')])   # (no groupby.apply: pandas>=3 drops the key column)
    df = pd.concat([keep_real, fk, df[(df.split != 'train') | (df.origin != 'ffpp')]]).reset_index(drop=True)

# hard checks
for a_, b_ in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    ia, ib = set(df[df.split == a_].source_id), set(df[df.split == b_].source_id)
    assert not (ia & ib), f'identity leak between {a_} and {b_}: {sorted(ia & ib)[:5]}'
for s_ in ('train', 'val', 'test'):
    assert set(df[df.split == s_].label) == {0, 1}, f'split {s_} lacks a class'
df.to_csv(OUT / 'manifest.csv', index=False)
print(pd.crosstab(df.split, df.label, margins=True)); print('\nper method:'); print(pd.crosstab(df.method, df.split))
print('\nlabel 0 = real, 1 = fake')
for p_, y_ in df.groupby(['origin', 'label']).head(2)[['path', 'label']].values:
    print('CHECK BY EYE ->', 'FAKE' if y_ else 'REAL', Path(p_).relative_to(BASE))

## 3. Data pipeline (preprocessing is an exact copy of DeepShield's inference preprocessing)

In [ ]:
IMN_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMN_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def project_preprocess(rgb_u8, size=IMG):
    """Same as ai_models.detectors.BaseDetector.preprocess for the chosen normalization, interpolation='bilinear'."""
    x = cv2.resize(rgb_u8, (size, size), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    if NORM == 'zero_one':
        x = x / 255.0
    elif NORM == 'minus_one_one':
        x = x / 127.5 - 1.0
    else:
        x = (x / 255.0 - IMN_MEAN) / IMN_STD
    return np.transpose(x, (2, 0, 1))

def augment(rgb, rng):
    """Mild, realistic augmentation (what uploaded photos go through: crops, JPEG, resize, blur, lighting)."""
    h, w = rgb.shape[:2]
    if rng.random() < 0.5:
        rgb = rgb[:, ::-1]
    if rng.random() < 0.7:                                   # random crop keeping 85-100% of each side
        s = rng.uniform(0.85, 1.0); ch, cw = int(h * s), int(w * s)
        y0, x0 = rng.integers(0, h - ch + 1), rng.integers(0, w - cw + 1)
        rgb = rgb[y0:y0 + ch, x0:x0 + cw]
    rgb = np.ascontiguousarray(rgb)
    if rng.random() < 0.15:                                  # down/up-scale
        f = rng.uniform(0.5, 0.9); hh, ww = rgb.shape[:2]
        rgb = cv2.resize(cv2.resize(rgb, (max(8, int(ww * f)), max(8, int(hh * f))), interpolation=cv2.INTER_AREA),
                         (ww, hh), interpolation=cv2.INTER_LINEAR)
    if rng.random() < 0.15:
        rgb = cv2.GaussianBlur(rgb, (0, 0), rng.uniform(0.1, 1.2))
    if rng.random() < 0.30:                                  # JPEG recompression
        ok, enc = cv2.imencode('.jpg', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(50, 96))])
        if ok: rgb = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    if rng.random() < 0.30:                                  # brightness / contrast
        rgb = np.clip(rgb.astype(np.float32) * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255).astype(np.uint8)
    return rgb

class FaceDataset(Dataset):
    def __init__(self, frame, train):
        self.paths, self.labels, self.train = frame.path.tolist(), frame.label.astype(int).tolist(), train
        self.rng = np.random.default_rng(SEED)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        bgr = cv2.imread(self.paths[i], cv2.IMREAD_COLOR)
        if bgr is None:
            raise RuntimeError(f'unreadable image: {self.paths[i]}')       # never skip silently
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        if self.train:
            rgb = augment(rgb, self.rng)
        return torch.from_numpy(project_preprocess(rgb)), self.labels[i]

def _worker_init(wid):
    cv2.setNumThreads(0)
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)
    ds = torch.utils.data.get_worker_info().dataset
    ds.rng = np.random.default_rng(s)

def make_loader(split, train):
    frame = df[df.split == split]
    return DataLoader(FaceDataset(frame, train), batch_size=BATCH, shuffle=train, drop_last=train,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'), worker_init_fn=_worker_init,
                      persistent_workers=NUM_WORKERS > 0)

train_loader, val_loader = make_loader('train', True), make_loader('val', False)
ytr = df[df.split == 'train'].label.values
class_w = torch.tensor([len(ytr) / (2 * (ytr == c).sum()) for c in (0, 1)], dtype=torch.float32)
print('train batches/epoch:', len(train_loader), '| class weights (real, fake):', class_w.tolist())

## 4. Model + training (AMP, warm-up + cosine, class-weighted loss, checkpoint every epoch, resumable)

In [ ]:
class Meso4(nn.Module):
    """MesoNet-4 (Afchar et al., 2018): 4 x (conv -> ReLU -> BN -> pool), dropout, FC16 (LeakyReLU 0.1), FC out.
    Original has 1 sigmoid output; here 2 logits (softmax = same function family) so DeepShield uses output_mode=logits."""
    def __init__(self, size, num_classes=2):
        super().__init__()
        def blk(i, o, k, p): return [nn.Conv2d(i, o, k, padding=k // 2), nn.ReLU(), nn.BatchNorm2d(o, eps=1e-3), nn.MaxPool2d(p)]
        self.features = nn.Sequential(*blk(3, 8, 3, 2), *blk(8, 8, 5, 2), *blk(8, 16, 5, 2), *blk(16, 16, 5, 4))
        flat = 16 * (size // 2 // 2 // 2 // 4) ** 2
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(flat, 16), nn.LeakyReLU(0.1),
                                        nn.Dropout(0.5), nn.Linear(16, num_classes))
    def forward(self, x): return self.classifier(self.features(x))
    def get_classifier(self): return self.classifier[-1]

def load_afchar_meso4(net, h5_path):
    """Convert the authors' Keras weights (Meso4_DF.h5) into this PyTorch module (exact, verified against a numpy forward).
    Keras: kernel (kh,kw,in,out), Flatten is HWC; PyTorch: (out,in,kh,kw), Flatten is CHW. The original 1-unit sigmoid head
    (1 = REAL, 0 = deepfake) becomes 2 logits [0, -z] so softmax(fake) == 1 - sigmoid(z), i.e. the same function."""
    try:
        import h5py
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py'], check=True); import h5py
    W = {}
    with h5py.File(h5_path, 'r') as f:
        f.visititems(lambda n, o: W.__setitem__(n, np.array(o)) if hasattr(o, 'shape') else None)
    def get(layer, name):
        k = [k for k in W if k.split('/')[0].startswith(layer) and k.endswith(name + ':0')]
        assert len(k) == 1, (layer, name, k); return W[k[0]]
    ordered = lambda prefix: sorted({k.split('/')[0] for k in W if k.startswith(prefix)}, key=lambda x: int(x.rsplit('_', 1)[1]))
    convs, bns, dense = ordered('conv2d_'), ordered('batch_normalization_'), ordered('dense_')
    assert len(convs) == 4 and len(bns) == 4 and len(dense) == 2, (convs, bns, dense)
    sd = {}
    ci = [i for i, m in enumerate(net.features) if isinstance(m, nn.Conv2d)]; bi = [i for i, m in enumerate(net.features) if isinstance(m, nn.BatchNorm2d)]
    for n, c, b in zip(range(4), convs, bns):
        sd[f'features.{ci[n]}.weight'] = torch.tensor(get(c, 'kernel').transpose(3, 2, 0, 1).copy())
        sd[f'features.{ci[n]}.bias'] = torch.tensor(get(c, 'bias'))
        for kn, tn in (('gamma', 'weight'), ('beta', 'bias'), ('moving_mean', 'running_mean'), ('moving_variance', 'running_var')):
            sd[f'features.{bi[n]}.{tn}'] = torch.tensor(get(b, kn))
    k1 = get(dense[0], 'kernel'); side = int(round((k1.shape[0] // 16) ** 0.5)); assert 16 * side * side == k1.shape[0]
    sd['classifier.2.weight'] = torch.tensor(k1.reshape(side, side, 16, 16).transpose(2, 0, 1, 3).reshape(k1.shape[0], 16).T.copy())
    sd['classifier.2.bias'] = torch.tensor(get(dense[0], 'bias'))
    k2, b2 = get(dense[1], 'kernel'), get(dense[1], 'bias')                    # (16,1), (1,)
    sd['classifier.5.weight'] = torch.tensor(np.stack([np.zeros(16, np.float32), -k2[:, 0]]).astype(np.float32))
    sd['classifier.5.bias'] = torch.tensor(np.array([0.0, -b2[0]], np.float32))
    missing = net.load_state_dict({k: v.float() for k, v in sd.items()}, strict=False)
    assert not missing.unexpected_keys and all('num_batches' in k for k in missing.missing_keys), missing
    return W

def build_model(pretrained, train_regularisation=True):
    if ARCH == 'mesonet4':
        net = Meso4(IMG, 2)
        if MESO_INIT == 'afchar' and (pretrained or MESO_H5):
            path = MESO_H5
            if not path:
                import urllib.request; path = str(OUT / 'Meso4_DF.h5'); urllib.request.urlretrieve(MESO_URL, path)
            assert IMG == 256, 'the authors\' weights are for 256x256 input (Dense layer is 1024 wide)'
            load_afchar_meso4(net, path); print('MesoNet-4: initialised from the authors\' Meso4_DF weights (fine-tuning)')
        else:
            print('MesoNet-4: random init (scratch)')
        return net
    kw = dict(pretrained=pretrained, num_classes=2)
    if ARCH == 'vit_b16': kw['img_size'] = IMG
    if train_regularisation:
        kw['drop_rate'] = 0.3
        if A['dpr']: kw['drop_path_rate'] = A['dpr']
    return timm.create_model(A['timm'], **kw)

model = build_model(pretrained=not SMOKE).to(DEVICE)
head_ids = {id(p) for p in model.get_classifier().parameters()}
opt = torch.optim.AdamW([
    {'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': LR_BACKBONE},
    {'params': [p for p in model.parameters() if id(p) in head_ids], 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
base_lrs = [g['lr'] for g in opt.param_groups]
total_steps, warm_steps = EPOCHS * len(train_loader), WARMUP_EPOCHS * len(train_loader)

def set_lr(step):
    f = (step + 1) / warm_steps if step < warm_steps else \
        0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (step - warm_steps) / max(1, total_steps - warm_steps)))
    for g, b in zip(opt.param_groups, base_lrs):
        g['lr'] = b * f

use_amp = DEVICE == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
criterion = nn.CrossEntropyLoss(weight=class_w.to(DEVICE), label_smoothing=LABEL_SMOOTHING)

N_FP32_FALLBACK = [0]

@torch.inference_mode()
def predict(loader):
    model.eval(); ps, ys = [], []; loss_sum = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        logits = logits.float()
        bad = ~torch.isfinite(logits).all(dim=1)
        if bad.any():                                        # fp16 overflow on a few images -> recompute those in fp32
            logits[bad] = model(x[bad]).float()
            N_FP32_FALLBACK[0] += int(bad.sum())
            if not torch.isfinite(logits).all():
                raise RuntimeError('non-finite logits even in fp32: a corrupt input image?')
        loss_sum += nn.functional.cross_entropy(logits, y, reduction='sum').item(); n += len(y)
        ps.append(torch.softmax(logits, 1)[:, 1].cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), loss_sum / n

history, best = [], {'auc': -1, 'loss': 9e9, 'epoch': -1}
start_epoch, bad_epochs, step = 0, 0, 0
if RESUME and (OUT / 'last.pt').exists():
    ck = torch.load(OUT / 'last.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
    history, best, start_epoch, bad_epochs = ck['history'], ck['best'], ck['epoch'] + 1, ck['bad_epochs']
    step = start_epoch * len(train_loader)
    print(f'RESUMED from epoch {start_epoch}, best val AUC so far {best["auc"]:.4f}')

for epoch in range(start_epoch, EPOCHS):
    model.train(); t0 = time.time(); run_loss = seen = 0
    for i, (x, y) in enumerate(train_loader):
        set_lr(step); step += 1
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            loss = criterion(model(x).float(), y)
        if not torch.isfinite(loss):
            raise RuntimeError(f'non-finite loss at epoch {epoch} step {i}; lower the learning rate')
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        run_loss += loss.item() * len(y); seen += len(y)
        if i % 200 == 0:
            print(f'  epoch {epoch + 1} step {i}/{len(train_loader)} loss {run_loss / seen:.4f}', flush=True)
    yv, pv, vloss = predict(val_loader)
    auc = roc_auc_score(yv, pv); bacc = balanced_accuracy_score(yv, pv >= 0.5)
    history.append({'epoch': epoch + 1, 'train_loss': run_loss / seen, 'val_loss': vloss, 'val_auc': auc,
                    'val_bal_acc@0.5': bacc, 'minutes': (time.time() - t0) / 60})
    print(f'EPOCH {epoch + 1}/{EPOCHS}: train {run_loss / seen:.4f} | val loss {vloss:.4f} AUC {auc:.4f} bal.acc {bacc:.4f} | '
          f'{(time.time() - t0) / 60:.1f} min', flush=True)
    improved = auc > best['auc'] + 1e-5
    if improved:
        best = {'auc': auc, 'loss': vloss, 'epoch': epoch + 1}; bad_epochs = 0
        torch.save(model.state_dict(), OUT / 'best.pt')
    else:
        bad_epochs += 1
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'history': history,
                'best': best, 'epoch': epoch, 'bad_epochs': bad_epochs}, OUT / 'last.pt')
    json.dump(history, open(OUT / 'history.json', 'w'), indent=1)
    if bad_epochs >= PATIENCE:
        print(f'early stop: no val AUC improvement for {PATIENCE} epochs'); break
print('best:', best)

try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    h = pd.DataFrame(history); fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='val'); ax[0].legend(); ax[0].set_title('loss')
    ax[1].plot(h.epoch, h.val_auc); ax[1].set_title('val AUC'); fig.tight_layout(); fig.savefig(OUT / 'curves.png', dpi=120)
except Exception as e:
    print('plot skipped:', e)

## 5. Final evaluation (run ONCE, after training is finished)
The decision threshold is chosen on the **validation** set and then applied unchanged to test.

In [ ]:
GUARD = OUT / '.test_evaluated'
if GUARD.exists() and not SMOKE:
    print('Test set was already evaluated in this run. Re-running would turn it into a tuning set; report the stored result in eval_report.json.')
    report = json.load(open(OUT / 'eval_report.json'))
else:
    model.load_state_dict(torch.load(OUT / 'best.pt', map_location=DEVICE, weights_only=True))
    yv, pv, _ = predict(val_loader)
    fpr, tpr, thr = roc_curve(yv, pv); thr_val = float(thr[np.argmax(tpr - fpr)])          # Youden J on validation
    test_loader = make_loader('test', False)
    yt, pt, _ = predict(test_loader)
    print('images recomputed in fp32 (fp16 overflow):', N_FP32_FALLBACK[0])
    assert np.isfinite(pt).all()
    test_df = df[df.split == 'test'].reset_index(drop=True)

    def metrics(y, p, t):
        pred = (p >= t).astype(int); tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='binary', zero_division=0)
        fpr_, fnr_ = fp / max(1, fp + tn), fn / max(1, fn + tp)
        f, t_, _ = roc_curve(y, p); eer = float(f[np.nanargmin(np.abs((1 - t_) - f))])
        return {'threshold': round(float(t), 4), 'auc': round(float(roc_auc_score(y, p)), 4),
                'average_precision': round(float(average_precision_score(y, p)), 4),
                'accuracy': round(float((pred == y).mean()), 4), 'balanced_accuracy': round(float(balanced_accuracy_score(y, pred)), 4),
                'precision': round(float(pr), 4), 'recall_fake': round(float(rc), 4), 'f1': round(float(f1), 4),
                'fpr': round(float(fpr_), 4), 'fnr': round(float(fnr_), 4), 'eer': round(eer, 4),
                'confusion': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}, 'n': int(len(y))}

    per_method = {}
    real_mask = (yt == 0)
    for m in sorted(test_df.method.unique()):
        if m == 'real' or m.endswith('_real'): continue
        mask = (test_df.method.values == m)
        per_method[m] = {'n': int(mask.sum()), 'detection_rate@val_thr': round(float((pt[mask] >= thr_val).mean()), 4)}
    per_origin = {}
    for o_ in sorted(test_df.origin.unique()):
        mk = (test_df.origin.values == o_)
        if len(set(yt[mk])) == 2:
            per_origin[o_] = metrics(yt[mk], pt[mk], thr_val)
        else:
            per_origin[o_] = {'n': int(mk.sum()), 'note': 'one class only', 'detection_rate@val_thr': round(float((pt[mk] >= thr_val).mean()), 4)}
    report = {'dataset': ('FF++ C23 faces + extra real/AI-face datasets (mixed)' if MIXED else 'FaceForensics++ C23 extracted faces (kaggle: gradientvoyager)'),
              'per_origin@val_thr': per_origin, 'split': 'identity-disjoint', 'seed': SEED,
              'val_threshold_youden': thr_val, 'best_epoch': best['epoch'], 'best_val_auc': round(best['auc'], 4),
              'test@0.5': metrics(yt, pt, 0.5), 'test@val_threshold': metrics(yt, pt, thr_val),
              'real_false_positive_rate@val_thr': round(float((pt[real_mask] >= thr_val).mean()), 4),
              'per_method': per_method,
              'caveats': ['Single-dataset result: FF++ C23 only. Cross-dataset performance (e.g. Celeb-DF) is expected to be lower and must be measured separately.',
                          'Frames of one video are highly correlated; effective sample size is the number of videos, not images.']
                         + (['MIXED mode: extra datasets have no identity ids, so they are split by image; near-duplicate images across splits cannot be excluded. Judge generalisation on Celeb-DF / the live-site images, not on the extra-data test numbers.'] if MIXED else [])}
    json.dump(report, open(OUT / 'eval_report.json', 'w'), indent=1)
    if not SMOKE: GUARD.write_text('done')
print(json.dumps(report, indent=1))

## 6. Export for DeepShield (ONNX + PyTorch state_dict) and prove parity with the project's inference path

In [ ]:
try:
    import onnx, onnxruntime as ort
except ModuleNotFoundError:                       # Kaggle image has no onnx runtime by default
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'])
    import onnx, onnxruntime as ort
net = build_model(False, False); net.load_state_dict(torch.load(OUT / 'best.pt', map_location='cpu', weights_only=True)); net.eval()

# (a) PyTorch state_dict (plain tensors; loads with weights_only=True; enables Grad-CAM in DeepShield)
pth = OUT / f'{KEY}_state_dict.pth'; torch.save(net.state_dict(), pth)

# (b) ONNX (CPU-friendly; no torch needed on the server)
onnx_path = OUT / f'{KEY}.onnx'
timm.layers.set_exportable(True)
dummy = torch.randn(1, 3, IMG, IMG)
kw = dict(input_names=['input'], output_names=['logits'], dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17)
try:
    torch.onnx.export(net, dummy, str(onnx_path), dynamo=False, **kw)
except TypeError:
    torch.onnx.export(net, dummy, str(onnx_path), **kw)
timm.layers.set_exportable(False)
onnx.checker.check_model(onnx.load(str(onnx_path)))

# (c) parity on real test images through the project's preprocessing: torch vs ONNX vs state_dict-reload (how DeepShield loads .pth)
reload_net = build_model(False, False); reload_net.load_state_dict(torch.load(pth, map_location='cpu', weights_only=True), strict=True); reload_net.eval()
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
sample = df[df.split == 'test'].sample(min(16, int((df.split == 'test').sum())), random_state=1)
worst_onnx = worst_pth = 0.0
for p_ in sample.path:
    rgb = cv2.cvtColor(cv2.imread(p_), cv2.COLOR_BGR2RGB); x = project_preprocess(rgb)[None]
    with torch.inference_mode():
        ref = torch.softmax(net(torch.from_numpy(x)), 1).numpy()[0]
        pth_out = torch.softmax(reload_net(torch.from_numpy(x)), 1).numpy()[0]
    onx = torch.softmax(torch.from_numpy(sess.run(None, {'input': x})[0]), 1).numpy()[0]
    worst_onnx = max(worst_onnx, float(np.abs(ref - onx).max())); worst_pth = max(worst_pth, float(np.abs(ref - pth_out).max()))
print(f'max |softmax diff|  torch vs ONNX: {worst_onnx:.2e}   torch vs reloaded state_dict: {worst_pth:.2e}')
assert worst_onnx < 1e-3 and worst_pth < 1e-6, 'export parity FAILED - do not use these files'
print('PARITY OK')

def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

training = {'dataset': ('FF++ C23 (gradientvoyager/faceforensics-c23-extracted-faces-100k) + extra: ' + ', '.join(sorted(o for o in df.origin.unique() if o != 'ffpp'))) if MIXED else 'FaceForensics++ C23 extracted faces (gradientvoyager/faceforensics-c23-extracted-faces-100k)',
            'dataset_version': 'kaggle', 'date': datetime.date.today().isoformat(),
            'parameters': {'arch': (f"timm {A['timm']} (ImageNet pretrained)" if A['timm'] else A['desc'] + (', fine-tuned from the authors\' Meso4_DF weights' if MESO_INIT == 'afchar' else ', trained from scratch') + ', 2-way head'), 'img': IMG, 'epochs_run': len(history), 'batch': BATCH,
                           'lr_backbone': LR_BACKBONE, 'lr_head': LR_HEAD, 'weight_decay': WEIGHT_DECAY, 'label_smoothing': LABEL_SMOOTHING,
                           'split': 'identity-disjoint 70/15/15', 'seed': SEED},
            'metrics': {'test_auc': report['test@val_threshold']['auc'], 'test_balanced_accuracy': report['test@val_threshold']['balanced_accuracy'],
                        'test_eer': report['test@val_threshold']['eer'], 'scope': ('mixed FF++ + extra test (see per_origin in eval_report.json)' if MIXED else 'FF++ C23 same-distribution test only')}}
common = {'input_size': IMG, 'normalization': NORM, 'output_mode': 'logits', 'fake_index': 1, 'num_outputs': 2,
          'interpolation': 'bilinear', 'threshold': round(report['val_threshold_youden'], 4), 'threshold_version': 'youden-val-v1',
          'version': '2.0.0' if MIXED else '1.0.0', 'training': training}
entry_onnx = {'file': f'{KEY}/model.onnx', 'framework': 'onnx', 'sha256': sha(onnx_path), **common}
entry_torch = {'file': f'{KEY}/model.pth', 'framework': 'torch', 'sha256': sha(pth), **common}
json.dump({KEY: entry_onnx}, open(OUT / 'models_entry_onnx.json', 'w'), indent=2)
if ARCH != 'mesonet4':                                   # DeepShield's .pth loader has no MesoNet architecture: use the ONNX file
    json.dump({KEY: entry_torch}, open(OUT / 'models_entry_torch.json', 'w'), indent=2)
print('\n=== models.json entry (ONNX, CPU server, no Grad-CAM) ===\n', json.dumps({KEY: entry_onnx}, indent=2))
if ARCH != 'mesonet4':
    print('\n=== models.json entry (PyTorch state_dict, enables Grad-CAM) ===\n', json.dumps({KEY: entry_torch}, indent=2))
print(f'\nDownload: {KEY}.onnx, {KEY}_state_dict.pth, models_entry_*.json, eval_report.json, manifest.csv, history.json, curves.png')